# Day 80 — Dataloaders

This notebook covers PyTorch DataLoaders (`torch.utils.data.DataLoader`), Mini-Batching (`batch_size`), Epoch Shuffling (`shuffle`), Remainder Batch Handling (`drop_last`), Worker Configuration (`num_workers`), and Dynamic Batch Size Hygiene

## Validation goals
- Understand how `DataLoader` wraps a `Dataset` to automate mini-batching, epoch shuffling, remainder handling, and parallel loading
- Configure distinct `DataLoader` settings for training (`shuffle=True`) versus validation and testing (`shuffle=False`)
- Integrate `DataLoader` iteration into multi-epoch training loops unpacking `(batch_X, batch_y)` pairs
- Implement dynamic batch size inspection (`xb.size(0)`) to prevent shape mismatch crashes and unweighted metric distortion on trailing partial batches

### 1. What DataLoader Adds on Top of a Dataset

A Dataset knows how many samples exist and how to fetch one sample by index. A DataLoader handles how samples are grouped and served during training or evaluation:
- Mini-Batching (`batch_size`): Groups individual (x, y) pairs into batch tensors of shape (`batch_size`, ...) so matrix operations run efficiently across batches.
- Shuffling (`shuffle`): Randomizes sample order each epoch during training so the model does not memorize row ordering or encounter identical batch sequences every pass.
- Drop Remainder (drop_last): Optionally drops a trailing partial batch if total samples are not evenly divisible by batch_size (helpful when fixed batch sizes are required).
- Parallel Loading (num_workers): Uses background worker processes to prepare batches while the main process runs forward/backward passes (keep num_workers=0 on Windows or constrained environments if multiprocessing causes spawn overhead or memory pressure).

### 2. Creating Training and Validation/Test DataLoaders

A standard rule of thumb:

- Training loader: shuffle=True so batches vary every epoch.
- Validation / Test loader: shuffle=False so evaluation order stays deterministic and reproducible across runs.

In [1]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset


class TabularDemandDataset(Dataset):
    """Custom PyTorch Dataset for tabular feature matrices and continuous targets."""

    def __init__(self, features: np.ndarray, targets: np.ndarray):
        self.X = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(targets, dtype=torch.float32).reshape(-1, 1)

    def __len__(self) -> int:
        return self.X.shape[0]

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]


# Synthetic split data matching our preprocessed tabular feature setup
rng = np.random.default_rng(42)
X_train_np = rng.normal(size=(100, 12))
y_train_np = rng.uniform(20.0, 400.0, size=(100,))

X_test_np = rng.normal(size=(25, 12))
y_test_np = rng.uniform(20.0, 400.0, size=(25,))

train_dataset = TabularDemandDataset(X_train_np, y_train_np)
test_dataset = TabularDemandDataset(X_test_np, y_test_np)

# Create DataLoaders (num_workers=0 is safe and predictable across OS setups)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)

print("--- Day 80: DataLoader Inspection ---")
print(f"Train samples: {len(train_dataset)}, Train batches (batch_size=16): {len(train_loader)}")
print(f"Test samples:  {len(test_dataset)}, Test batches (batch_size=16):  {len(test_loader)}")

for batch_idx, (batch_X, batch_y) in enumerate(train_loader):
    print(f"Batch {batch_idx} -> X shape: {batch_X.shape}, y shape: {batch_y.shape}")
    if batch_idx == 2:
        break

--- Day 80: DataLoader Inspection ---
Train samples: 100, Train batches (batch_size=16): 7
Test samples:  25, Test batches (batch_size=16):  2
Batch 0 -> X shape: torch.Size([16, 12]), y shape: torch.Size([16, 1])
Batch 1 -> X shape: torch.Size([16, 12]), y shape: torch.Size([16, 1])
Batch 2 -> X shape: torch.Size([16, 12]), y shape: torch.Size([16, 1])


Notice that with 100 training samples and `batch_size=16`, len(`train_loader`) is 7 (6 full batches of 16 plus a final batch of 4 samples, since `drop_last` defaults to `False`).

### 3. Integrating DataLoader Into an Epoch Loop

In training scripts, each epoch iterates over train_loader once, unpacking (`batch_X, batch_y`) on every step:

In [2]:
import torch
from torch.utils.data import DataLoader, TensorDataset

X = torch.randn((32, 6), dtype=torch.float32)
y = torch.randn((32, 1), dtype=torch.float32)
loader = DataLoader(TensorDataset(X, y), batch_size=8, shuffle=True)

epochs = 3
for epoch in range(1, epochs + 1):
    epoch_batches = 0
    total_rows = 0

    for batch_X, batch_y in loader:
        epoch_batches += 1
        total_rows += batch_X.shape[0]
        # Tomorrow (Day 81): pass batch_X into nn.Module and compute loss against batch_y

    print(f"Epoch {epoch}: processed {epoch_batches} batches ({total_rows} rows)")

Epoch 1: processed 4 batches (32 rows)
Epoch 2: processed 4 batches (32 rows)
Epoch 3: processed 4 batches (32 rows)


### 4. Practical Engineering Gotchas

Don't Shuffle Test/Validation Loaders: Keeping `shuffle=False` on validation and test loaders makes debugging and metric comparisons deterministic.
- Watch the Last Batch Shape: When dataset size is not a multiple of `batch_size`, the final batch is smaller (e.g., (4, 12) instead of (16, 12)). Avoid hardcoding 16 inside your model or loss logic—always rely on dynamic tensor shapes like `batch_X.shape[0]`.
- Keep num_workers Measured: Extra worker processes add IPC overhead on small tabular datasets or tight memory/CPU budgets; start with num_workers=0 and benchmark before increasing.

### Guided Question:

Why do we typically set `shuffle=True` for `train_loader` but `shuffle=False` for `test_loader`?

### Answer:

Setting `shuffle=True` for train_loader and shuffle=False for test_loader matches the different goals of learning versus evaluating.

**1. Why `shuffle=True` for train_loader**

During training, your model updates its weights after every batch using mini-batch gradient descent. Shuffling the data at the start of every epoch prevents subtle learning traps:
- Avoids batch bias & order artifacts: If your training data is sorted (for example, all Class 0 samples first, then Class 1, or sorted by time/difficulty), unshuffled batches give each gradient step a skewed view of the data. Shuffling creates diverse mini-batches so gradient updates stay noisy in a healthy, unbiased way rather than oscillating toward one group at a time.
- Prevents memorization of order / batch patterns: Models can sometimes pick up spurious patterns if samples appear in a fixed sequence or if normalization/batch statistics behave strangely across ordered batches.
- Fresh combinations each epoch: Re-shuffling every epoch means a sample that shared a batch with one set of items in Epoch 1 meets different neighbors in Epoch 2, helping generalization.

(Exception: For true time-series forecasting where temporal order carries meaning, or iterable/streaming datasets, standard random row shuffling isn't always appropriate—you'd use windowing/sequence datasets or time-aware splits instead.)

**2. Why shuffle=False for test_loader**

During testing (and validation), you aren't updating weights—you're measuring performance. Shuffling adds overhead without improving accuracy:
- Deterministic, reproducible evaluation: Keeping shuffle=False means every evaluation run processes samples in the exact same order, making metrics, logs, and debugging consistent and repeatable.
- Aligns predictions with original sample order: If you collect predictions (preds) across batches to compute custom metrics, plot error curves, inspect misclassified examples, or match predictions back to IDs/rows in a test table, shuffle=False keeps everything in a predictable order.
- Avoids unnecessary work: Shuffling indices each pass isn't needed when you just want one clean pass over the test set.

**Summary**

- train_loader (`shuffle=True`): Better gradients, less order bias, stronger generalization across epochs.
- test_loader (`shuffle=False`): Reproducible results, easy alignment with original samples, no wasted shuffling.

### Guided Question:

If a validation dataset has 45 samples and DataLoader uses `batch_size=16` with `drop_last=False`, how many batches are yielded per pass and what are the sizes of those batches?

### Answer:

A DataLoader with 45 samples, `batch_size=16`, and drop_last=False yields 3 batches per pass with sizes 16, 16, and 13.

**How It Works**

- Full batches (`batch_size=16`): Dividing 45 by 16 gives 2 full batches (2 × 16 = 32 samples).
- Remainder / final batch: Subtracting those 32 samples from 45 leaves 13 samples (45 - 32 = 13).
- Effect of `drop_last=False`: Because drop_last is False, PyTorch keeps that smaller last batch instead of discarding it—which is standard for validation and testing so every sample gets evaluated.

### Guided Question:

Why is writing code that assumes every batch has shape (`batch_size`, features) risky when `drop_last=False`?

### Answer:

When `drop_last=False`, the final batch of an epoch is often smaller than `batch_size—it` has shape (`remainder, features`) rather than (`batch_size, features`). Hardcoding or assuming the first dimension is always batch_size can break your loop or silently distort metrics.

**1. Shape Mismatches & Runtime Crashes**

If you hardcode batch_size inside your loop—for example, reshaping a tensor with .view(`batch_size, ...`) or indexing up to batch_size—the last smaller batch will fail with a shape or index error (such as RuntimeError: shape `'[32, ...]' is invalid for input of size ...`).

**2. Unweighted or Distorted Metric Averages**

Suppose you accumulate loss or accuracy per batch and divide by the number of batches at the end of an epoch:
- If 10 batches have 32 samples and the last batch has only 5 samples, taking a simple average across all 11 batches treats that 5-sample batch as equal in weight to a full 32-sample batch.
- Instead, compute sums weighted by the actual batch size (x.size(0)) and divide by the total number of samples (len(dataset)).

**3. Custom Logic & Feature Engineering Assumptions**

If you write custom batch-level operations—such as splitting a batch in half, pairing samples across a batch, or building batch-specific masks assuming an even or fixed count—a smaller final batch (especially a batch of size 1) can produce unexpected shapes, empty slices, or edge-case bugs.

**Safer Pattern**

Always derive the current batch size dynamically from the tensor itself rather than assuming batch_size:

In [ ]:
for xb, yb in loader:
    current_batch_size = xb.size(0)  # equals batch_size most of the time, or remainder on the last batch
    ...